# Step 01. Cubic Spline 경로 생성

> **author**: Minwon Lee, Hanyang University
>
> **참고**: PythonRobotics (https://github.com/AtsushiSakai/PythonRobotics)
>
> **교육 주관**: Hanyang University, Automotive Intelligence Laboratory (AILAB)

전역 경로는 띄엄띄엄한 점(waypoint)으로 주어지므로, 이 점들을 부드러운 곡선으로 이어주는 3차 스플라인(cubic spline)을 알아봅니다.

## 학습 목표

1. 점을 직선으로 이을 때 생기는 방향 꺾임(불연속) 확인
2. 각 구간을 3차함수 $a + b\Delta + c\Delta^2 + d\Delta^3$ 로 표현
3. 접점(구간과 구간이 만나는 점)의 부드러움(C0 / C1 / C2 연속) 조건 이해
4. 라이브러리가 푼 계수로 각 구간 3차함수 값을 직접 계산 (TODO 1)
5. 기울기(1차 미분)를 직접 계산해 접점에서 기울기 연속 확인 → C1 (TODO 2)
6. 구간별 3차함수를 이어 만든 cubic spline 시각화 (접점에서 값이 이어짐 → C0)
7. 점 이동에 따른 spline 변화 관찰 (interactive)

In [ ]:
%matplotlib inline
import sys, os, math
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('../utils/02_trajectory_planning'))

import numpy as np
import matplotlib.pyplot as plt

from step01_cubic_spline_path import CubicSpline1D
from step01_cubic_spline_path import (
    PlotStraightLineLimit,
    PlotSegmentDefinition,
    PlotSlopeComparison,
    PlotCubicSpline,
    InteractiveSpline,
)

print('ready')

## 1. 직선 연결의 한계

점들을 직선으로 이으면 코너마다 방향이 꺾입니다. 오른쪽 그래프의 방향각(heading)은 구간 안에서는 일정하다가 코너에서 불연속적으로 점프합니다.  
차가 이 경로를 따라가려면 코너마다 방향을 급하게 바꿔야 하므로, 부드러운 곡선이 필요합니다.

In [ ]:
# 전역 경로 점 7개 (waypoint): 차가 지나가야 할 목표 좌표
x = [0.0, 2.0, 4.0, 6.0, 8.0, 10.0, 12.0]   # 각 점의 x 좌표
y = [1.0, 3.5, 5.0, 4.0, 6.0,  5.0,  3.0]   # 각 점의 y 좌표

PlotStraightLineLimit(x, y)

## 2. 구간별 3차함수 표현

이웃한 두 점 사이(한 구간)를 3차함수 하나로 표현합니다. 시작점 $x_i$ 기준 거리를 $\Delta = x - x_i$ 로 두면

$$S_i(\Delta) = a_i + b_i\,\Delta + c_i\,\Delta^2 + d_i\,\Delta^3$$

구간마다 계수 네 개 $a, b, c, d$ 가 곡선 한 조각을 결정합니다. 점 7개 → 구간 6개 → 계수 $6 \times 4 = 24$ 개입니다.

아래 그림으로 구조를 확인합니다. 왼쪽은 색이 다른 각 조각(3차함수 $S_0, S_1, \dots$)이고, 오른쪽은 접점(구간과 구간이 만나는 점, 빨간 원) $x_1$ 에서 맞춰야 할 연속 조건입니다. 수식의 $S_0(x_1)$ 은 "접점 $x_1$ 에서 조각 $S_0$ 의 값"이라는 뜻입니다.

In [ ]:
PlotSegmentDefinition(x, y)

## 3. 부드러움 조건 (C0 / C1 / C2)

계수 24개를 정하려면 그만큼의 조건이 필요합니다. 다음 세 가지를 요구합니다.

1. 값 통과 (C0): 각 조각이 양 끝 점을 지남 → 경로가 끊기지 않음 (조각당 2개, $6 \times 2 = 12$개)
2. 1차 미분(기울기) 연속 (C1): 접점에서 들어오는 기울기 = 나가는 기울기 → 방향이 꺾이지 않음 (접점 5곳 = 5개)
3. 2차 미분(휘는 정도) 연속 (C2): 접점에서 휘는 정도가 같음 → 곡률이 튀지 않음(승차감) (접점 5곳 = 5개)

여기까지 $12 + 5 + 5 = 22$개이고, 남은 2개는 양 끝 경계조건(끝에서 휘지 않음)으로 채워 계수 24개가 결정됩니다. 계수를 푸는 과정은 라이브러리에 맡깁니다.

## 4. [TODO 1] 3차함수 조각 직접 계산

구간 $i$ 는 시작점 $x_i$ 로부터의 거리 $\Delta = x - x_i$ 에 대해

$$S_i(\Delta) = a_i + b_i\,\Delta + c_i\,\Delta^2 + d_i\,\Delta^3$$

계수 $a, b, c, d$ 는 라이브러리(`CubicSpline1D`)가 푼 값을 사용하므로, 위 공식 한 줄을 코드로 옮기면 됩니다.

아래 `EvaluateCubicSegment` 의 TODO 1 한 줄을 작성합니다. 이 함수가 섹션 6·7 의 그림을 직접 그립니다.

In [ ]:
def EvaluateCubicSegment(a, b, c, d, delta):
    # a, b, c, d : 이 구간(3차함수 조각)의 계수 -- 라이브러리가 풀어 준 값
    # delta      : 구간 시작점 x_i 로부터의 거리 (delta = x - x_i)
    ##### TODO 1 #####
    # 3차함수 조각 값 계산하기: 섹션 4 의 공식 S_i(Δ)
    # (라이브러리 함수 없이 사칙연산과 거듭제곱(**)만 사용)
    #################

    position = ...

    #################
    return position

print('EvaluateCubicSegment defined')

In [ ]:
spline = CubicSpline1D(x, y)   # spline: 라이브러리가 만든 3차 스플라인 객체 (구간별 계수 a,b,c,d 보유)

# 구간별 계수 (라이브러리)
# a 열이 waypoint 의 y 값과 같음에 주목: Δ=0 에서 S_i(0) = a_i, 즉 각 조각이 시작점을 정확히 통과
print('seg |     a       b       c       d')
print('----+--------------------------------')
for i in range(len(x) - 1):
    print(f' S{i} | {spline.a[i]:7.3f} {spline.b[i]:7.3f} {spline.c[i]:7.3f} {spline.d[i]:7.3f}')

# 라이브러리 결과(calc_position)와 비교
max_error = 0.0
for i in range(len(x) - 1):
    for query_x in np.linspace(x[i], x[i + 1], 50):
        my_value  = EvaluateCubicSegment(spline.a[i], spline.b[i], spline.c[i], spline.d[i], query_x - x[i])
        reference = spline.calc_position(query_x)
        max_error = max(max_error, abs(my_value - reference))

print(f'\nmax error vs library = {max_error:.2e}')
assert max_error < 1e-12, 'TODO 1 확인: 3차함수 값이 라이브러리와 불일치'
print('OK: 3차함수 계산이 라이브러리와 일치')

## 5. [TODO 2] 기울기(방향) 직접 계산

부드러움의 핵심은 접점에서의 기울기 연속(C1)입니다. 3차함수 $S_i(\Delta) = a + b\Delta + c\Delta^2 + d\Delta^3$ 를 $\Delta$ 로 한 번 미분하면

$$S_i'(\Delta) = b_i + 2c_i\,\Delta + 3d_i\,\Delta^2$$

아래 TODO 2 한 줄을 작성한 뒤, 경로를 따라 기울기가 어떻게 변하는지 직선과 비교합니다. 경로의 방향각은 $\text{heading} = \arctan(\text{기울기})$ 이므로, 기울기가 연속이면 섹션 1 에서 본 방향각 점프도 사라집니다.

In [ ]:
def EvaluateCubicSlope(a, b, c, d, delta):
    # a, b, c, d : 구간(3차함수 조각)의 계수 / delta : 구간 시작점으로부터의 거리
    ##### TODO 2 #####
    # 기울기(1차 미분) 계산하기: 섹션 5 의 공식 S_i'(Δ)
    # (라이브러리 함수 없이 사칙연산과 거듭제곱(**)만 사용)
    #################

    slope = ...

    #################
    return slope

# 라이브러리 결과(calc_first_derivative)와 비교
spline = CubicSpline1D(x, y)   # spline: 라이브러리가 만든 3차 스플라인 객체
max_error = 0.0
for i in range(len(x) - 1):
    for query_x in np.linspace(x[i], x[i + 1], 50):
        my_value  = EvaluateCubicSlope(spline.a[i], spline.b[i], spline.c[i], spline.d[i], query_x - x[i])
        reference = spline.calc_first_derivative(query_x)
        max_error = max(max_error, abs(my_value - reference))
print(f'max error vs library = {max_error:.2e}')
assert max_error < 1e-12, 'TODO 2 확인: 기울기가 라이브러리와 불일치'
print('OK: 기울기 계산이 라이브러리와 일치')

In [ ]:
PlotSlopeComparison(x, y, EvaluateCubicSlope)

## 6. Cubic Spline 시각화

이제 완성된 곡선을 확인합니다. 여러분이 작성한 `EvaluateCubicSegment` 로 각 구간 조각을 서로 다른 색으로 그립니다. 색이 바뀌는 곳이 접점인데, 회색 직선과 달리 값이 끊기지 않고(C0) 매끄럽게 이어집니다. 기울기 연속(C1)은 5번에서 확인했고, C2(휘는 정도 연속)는 다음 노트북(step02) 섹션 1 에서 곡률 그래프가 끊기지 않고 이어지는 것으로 확인합니다.

In [ ]:
PlotCubicSpline(x, y, EvaluateCubicSegment)

## 7. 점 이동에 따른 spline 변화 (interactive)

점 하나를 위아래로 움직이면 그 점을 지나도록 spline 이 다시 생성됩니다. 슬라이더로 점 번호와 높이 y 를 조절해 보세요. (셀을 실행하면 슬라이더가 나타납니다. 실행 전 미리보기에서는 한 장면만 보입니다.)

In [ ]:
InteractiveSpline(x, y, EvaluateCubicSegment)